# Predictive Modelling for Sales Forecasting
## Benchmarking Lasso, PCR, and Random Forest on a 25-Firm Panel

This notebook reproduces all key results. Source code is in `src/`.

**Pipeline:**
1. Load data and construct one-period-ahead target
2. Chronological 80/20 train/test split
3. Time-aware expanding-window CV (5 folds)
4. Lasso → optimal λ* via CV
5. PCR → 95% variance threshold → OLS
6. Random Forest → GridSearchCV
7. Extension: Lasso-PCR joint (λ, k) selection
8. Feature importance comparison and Spearman rank agreement
9. Bias-variance decomposition and residual analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import spearmanr

from src.utils import fmse, FEATURE_COLS, RANDOM_STATE, save_fig
from src.cv import PanelTimeSeriesCV
from src.data_loader import load_data, chronological_split, get_arrays
from src.models import train_lasso, train_pcr, train_rf, train_lasso_pcr

print('Imports OK')

## 1. Data

In [ ]:
df = load_data('data/DS2.xlsx')
train_df, test_df = chronological_split(df, train_ratio=0.80)
X_train_raw, y_train, X_test_raw, y_test = get_arrays(train_df, test_df)

## 2. Time-Aware Cross-Validation Setup

In [ ]:
tscv = PanelTimeSeriesCV(n_splits=5, date_array=train_df['date'].values)

print(f'Training set: {len(train_df["date"].unique())} unique dates')
dummy_X = np.zeros((len(train_df), 1))
for k, (tr, va) in enumerate(tscv.split(dummy_X)):
    tr_dates  = np.sort(np.unique(train_df['date'].values[tr]))
    val_dates = np.sort(np.unique(train_df['date'].values[va]))
    print(f'  Fold {k+1}: Train {pd.Timestamp(tr_dates[0]).date()} → '
          f'{pd.Timestamp(tr_dates[-1]).date()} '
          f'({len(tr_dates)} dates)  |  '
          f'Val {pd.Timestamp(val_dates[0]).date()} → '
          f'{pd.Timestamp(val_dates[-1]).date()} '
          f'({len(val_dates)} dates)')

## 3. Model Training

In [ ]:
print('--- Lasso ---')
lasso_res = train_lasso(train_df, test_df, tscv)

print('\n--- PCR ---')
pcr_res = train_pcr(train_df, test_df, lasso_res)

print('\n--- Random Forest ---')
rf_res = train_rf(train_df, test_df, tscv)

print('\n--- Lasso-PCR ---')
lpca_res = train_lasso_pcr(train_df, test_df, lasso_res, tscv)

## 4. Results Summary

In [ ]:
summary = pd.DataFrame({
    'Model'         : ['Lasso', 'PCR', 'Random Forest', 'Lasso-PCR'],
    'Train FMSE'    : [lasso_res['train_fmse'], pcr_res['train_fmse'],
                       rf_res['train_fmse'],    lpca_res['train_fmse']],
    'Test FMSE'     : [lasso_res['test_fmse'],  pcr_res['test_fmse'],
                       rf_res['test_fmse'],     lpca_res['test_fmse']],
})
summary['Gen. Gap'] = summary['Test FMSE'] - summary['Train FMSE']
print(summary.to_string(index=False))

## 5. Feature Importance and Rank Agreement

In [ ]:
lasso_coef   = lasso_res['model'].coef_
all_names    = lasso_res['all_names']
lasso_coef_z = pd.Series(
    {f: abs(c) for f, c in zip(all_names, lasso_coef) if f.startswith('z')},
    name='Lasso |coef|')

rf_gini_z = rf_res['imp_gini'][[f for f in rf_res['imp_gini'].index
                                  if f.startswith('z')]].rename('RF Gini')

l_arr = lasso_coef_z.reindex(FEATURE_COLS).fillna(0).values
r_arr = rf_gini_z.reindex(FEATURE_COLS).fillna(0).values
rho, pval = spearmanr(l_arr, r_arr)

print(f'Spearman rho (Lasso |coef| vs RF Gini): {rho:.3f}  (p = {pval:.3f})')

rank_df = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Lasso rank': pd.Series(l_arr).rank(ascending=False).values,
    'RF rank'   : pd.Series(r_arr).rank(ascending=False).values,
})
rank_df['Rank diff'] = (rank_df['Lasso rank'] - rank_df['RF rank']).abs()
print('\nLargest rank disagreements:')
print(rank_df.nlargest(5, 'Rank diff').to_string(index=False))

## 6. FMSE Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue', 'darkorange', 'forestgreen', 'mediumpurple']
bars   = ax.bar(summary['Model'], summary['Test FMSE'],
                color=colors, edgecolor='white', width=0.5)
ax.bar_label(bars,
             labels=[f'{v:.2f}' for v in summary['Test FMSE']],
             padding=4, fontsize=10, fontweight='bold')
ax.set_ylabel('Out-of-sample FMSE', fontsize=12)
ax.set_title('Model Comparison — Out-of-sample FMSE (lower is better)', fontsize=12)
ax.grid(True, alpha=0.25, axis='y')
ax.set_ylim(0, max(summary['Test FMSE']) * 1.15)
plt.tight_layout()
save_fig(fig, 'model_comparison_fmse.png')
plt.show()